In [ ]:
from pathlib import Path

import polars as pl
from polars import selectors as cs

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_float("full")


polars.config.Config

In [2]:
data_assemblies = pl.concat(
    pl.read_csv(
        p / "quast_assembly.csv",
        infer_schema_length=None,
        null_values=[""],
    )
    for p in sorted(Path("../data").iterdir())
    if p.is_dir()
)

In [3]:
data_bins = pl.concat(
    [
        pl.read_csv(
            p / "bin_summary.tsv",
            separator="\t",
            infer_schema_length=None,
            null_values=[""],
        )
        .with_columns(
            dataset=pl.lit(p.name).str.split("_").list.get(0),
            experiment=pl.lit(p.name).str.split("_").list.get(1)
        )
        .drop(cs.starts_with("Depth"))
        for p in sorted(Path("../data").iterdir())
        if p.is_dir()
    ],
    how="diagonal_relaxed",
).with_columns(assembler=pl.col("bin").str.split("-").list.get(0))

In [4]:
# Summary of the assemblies per dataset and assembler
data_assemblies.group_by(["dataset", "experiment", "assembler"]).agg(
    mean_length=pl.col("Total length").mean().round(0),
    mean_n_contigs=pl.col("# contigs (>= 0 bp)").mean().round(0),
    mean_n50=pl.col("N50").mean().round(0),
).sort(["dataset", "experiment", "assembler"])

dataset,experiment,assembler,mean_length,mean_n_contigs,mean_n50
str,str,str,f64,f64,f64
"""maghini""","""hybrid""","""FLYE""",300969451,5648,200725
"""maghini""","""hybrid""","""MEGAHIT""",375361673,238407,11842
"""maghini""","""hybrid""","""METAMDBG""",328805771,7043,178235
"""maghini""","""hybrid""","""SPAdes""",375210976,466572,11970
"""maghini""","""hybrid""","""SPAdesHybrid""",414451863,386372,34351
"""maghini""","""polish""","""FLYE""",300846090,5648,200751
"""maghini""","""polish""","""METAMDBG""",328865994,7043,178003
"""zymo""","""hybrid""","""FLYE""",554088862,12174,106134
"""zymo""","""hybrid""","""MEGAHIT""",549466815,544106,3990


In [5]:
# Summary of the bins per dataset and assembler
data_bins.group_by(["dataset", "experiment", "assembler"]).agg(
    number_of_bins=pl.len(), mean_length=pl.col("Total length_quast").mean().round(0)
).sort(["dataset", "experiment", "assembler"])

dataset,experiment,assembler,number_of_bins,mean_length
str,str,str,u32,f64
"""maghini""","""hybrid""","""FLYE""",8212,2164513
"""maghini""","""hybrid""","""MEGAHIT""",8279,2151079
"""maghini""","""hybrid""","""METAMDBG""",9233,2079741
"""maghini""","""hybrid""","""SPAdes""",8231,2170375
"""maghini""","""hybrid""","""SPAdesHybrid""",9780,2207746
"""maghini""","""polish""","""FLYE""",7403,2092391
"""maghini""","""polish""","""METAMDBG""",8479,1970973
"""zymo""","""hybrid""","""FLYE""",1611,1902286
"""zymo""","""hybrid""","""MEGAHIT""",979,2234695
